# Chapter 8: Dictionaries and Maps

This chapter demonstrates how a dictionary maps keys to values, using Roman numeral conversion as a practical example.

## Running the examples

Select the **JavaScript (Node.js)** kernel installed for this project, then choose **Run All Cells**. These code cells are executable JavaScript adaptations of the TypeScript examples so they run with the project's IJavascript setup. Type annotations and interfaces are compile-time TypeScript features and do not change the demonstrated runtime behavior.

## Integer to Roman numerals with a dictionary

The conversion table pairs each Roman numeral token with its numeric value. It includes the six subtractive forms (`IV`, `IX`, `XL`, `XC`, `CD`, and `CM`) alongside the ordinary symbols. The entries are listed from largest value to smallest so the algorithm can make a greedy choice: repeatedly take the largest token that fits, then subtract its value from the remaining number.

For example, `1994` is handled as `1000 + 900 + 90 + 4`, producing `M + CM + XC + IV`, or `MCMXCIV`. The table has a fixed size, and the input range is bounded, so the work and output size are bounded as well.

### Object dictionary and native `Map`

The first implementation uses an object as a dictionary: each numeral string is a key and its numeric value is the associated value. `Object.entries` gives the key/value pairs in their insertion order; because the numeral keys are ordinary strings rather than integer-like property names, that preserves the descending order needed by this algorithm. The second implementation uses JavaScript's native `Map`, whose iteration follows insertion order directly.

The original TypeScript version uses a `RomanNumeralMap` type and a `TestCase` interface to check the table and test data at compile time. The executable cell below is JavaScript, so it keeps the same conversion logic and examples without compile-time annotations.

In [1]:
(() => {
console.log('=== Integer to Roman Conversion (TypeScript) Demo ===\n');

                                                 

function intToRoman(num        )         {
  if (!Number.isInteger(num) || num < 1 || num > 3999) {
    throw new RangeError('Number must be an integer between 1 and 3999');
  }

  const romanMap                  = {
    M: 1000, CM: 900, D: 500, CD: 400, C: 100,
    XC: 90, L: 50, XL: 40, X: 10, IX: 9, V: 5, IV: 4, I: 1
  };
  let result = '';
  for (const romanNum in romanMap) {
    while (num >= romanMap[romanNum]) {
      result += romanNum;
      num -= romanMap[romanNum];
    }
  }
  return result;
}

function intToRomanWithMap(num        )         {
  if (!Number.isInteger(num) || num < 1 || num > 3999) {
    throw new RangeError('Number must be an integer between 1 and 3999');
  }
  const romanMap = new Map                ([
    ['M', 1000], ['CM', 900], ['D', 500], ['CD', 400], ['C', 100],
    ['XC', 90], ['L', 50], ['XL', 40], ['X', 10], ['IX', 9],
    ['V', 5], ['IV', 4], ['I', 1]
  ]);
  let result = '';
  for (const [romanNum, value] of romanMap) {
    while (num >= value) {
      result += romanNum;
      num -= value;
    }
  }
  return result;
}

                                                      
const testCases             = [
  { input: 3, expected: 'III' },
  { input: 58, expected: 'LVIII' },
  { input: 1994, expected: 'MCMXCIV' },
  { input: 2024, expected: 'MMXXIV' },
  { input: 3888, expected: 'MMMDCCCLXXXVIII' }
];

for (const { input, expected } of testCases) {
  const result = intToRoman(input);
  console.log(`${result === expected ? '✓' : '✗'} ${input} => ${result} (expected: ${expected})`);
}

console.log('\nObject and Map versions agree:', intToRoman(1994) === intToRomanWithMap(1994));
for (const num of [0, 4000, 2.5]) {
  try {
    intToRoman(num);
  } catch (error) {
    if (error instanceof RangeError) console.log(`Rejected ${num}: ${error.message}`);
  }
}
})();

=== Integer to Roman Conversion (TypeScript) Demo ===



✓ 3 => III (expected: III)


✓ 58 => LVIII (expected: LVIII)


✓ 1994 => MCMXCIV (expected: MCMXCIV)


✓ 2024 => MMXXIV (expected: MMXXIV)


✓ 3888 => MMMDCCCLXXXVIII (expected: MMMDCCCLXXXVIII)



Object and Map versions agree: true


Rejected 0: Number must be an integer between 1 and 3999


Rejected 4000: Number must be an integer between 1 and 3999


Rejected 2.5: Number must be an integer between 1 and 3999


## A reusable generic dictionary

The Roman numeral table was a fixed dictionary. This next example builds a reusable `Dictionary<V>` class with operations for setting, looking up, removing, and iterating over entries. The generic type parameter `V` lets TypeScript enforce a consistent value type: a `Dictionary<number>`, for example, accepts numeric values.

The internal object stores values by stringified keys, while `_size` tracks the number of entries. Setting a key already present updates its value without increasing the count. `keys()`, `values()`, and `forEach()` visit the stored entries, so they take O(n) time; direct lookup and updates are expected O(1) for ordinary object properties.

### Key and value constraints

This implementation intentionally rejects `null` and `undefined` values, so `hasKey` can use a non-null stored value to distinguish an entry from a missing key. Keys are converted to strings: numeric key `1` and string key `'1'` therefore refer to the same entry, and object keys are represented by JSON text. Distinct objects with the same JSON representation can collide. This is a simple teaching dictionary; native `Map` is a better fit when keys must retain their original types or object identity.

The implementation below uses a null-prototype object for storage, so keys such as `toString` do not collide with inherited properties. TypeScript generics and interfaces describe the types checked by the compiler; they do not change the runtime representation.

In [2]:
(() => {
/**
 * Dictionary (Map) Data Structure Implementation in TypeScript
 * 
 * A dictionary stores key-value pairs where each key is unique.
 * This TypeScript implementation provides type safety through generics.
 * 
 * @template V The type of values stored in the dictionary
 */

class Dictionary    {
  // Private property to store key-value pairs
          items                       = Object.create(null)                        ;
  
  // Private property to track the number of elements
          _size         = 0;

  /**
   * Converts any data type to a string representation
   * @private
   */
          elementToString(data         )         {
    if (data === null) {
      return 'null';
    } else if (data === undefined) {
      return 'undefined';
    } else if (typeof data === 'object') {
      return JSON.stringify(data);
    } else {
      return String(data);
    }
  }

  /**
   * Checks whether a key exists in the dictionary
   * Time Complexity: O(1)
   * 
   * @param key - The key to search for
   * @returns True if the key exists, false otherwise
   */
  hasKey(key                          )          {
    return Object.prototype.hasOwnProperty.call(this.items, this.elementToString(key));
  }

  /**
   * Adds or updates a key-value pair in the dictionary
   * Time Complexity: O(1)
   * 
   * @param key - The key to set
   * @param value - The value to associate with the key
   * @returns True if successful, false if key or value is invalid
   */
  set(key                          , value   )          {
    if (key != null && value != null) {
      const stringKey = this.elementToString(key);
      
      // Only increment size if this is a new key
      if (!this.hasKey(key)) {
        this._size++;
      }
      
      this.items[stringKey] = value;
      return true;
    }
    return false;
  }

  /**
   * Removes a key-value pair from the dictionary
   * Time Complexity: O(1)
   * 
   * @param key - The key to remove
   * @returns True if the key was found and removed, false otherwise
   */
  remove(key                          )          {
    if (this.hasKey(key)) {
      const stringKey = this.elementToString(key);
      delete this.items[stringKey];
      this._size--;
      return true;
    }
    return false;
  }

  /**
   * Retrieves the value associated with a key
   * Time Complexity: O(1)
   * 
   * @param key - The key to look up
   * @returns The value associated with the key, or undefined if not found
   */
  get(key                          )                {
    return this.items[this.elementToString(key)];
  }

  /**
   * Removes all key-value pairs from the dictionary
   * Time Complexity: O(1)
   */
  clear()       {
    this.items = Object.create(null)                        ;
    this._size = 0;
  }

  /**
   * Returns the number of key-value pairs in the dictionary
   */
  get size()         {
    return this._size;
  }

  /**
   * Checks if the dictionary is empty
   */
  isEmpty()          {
    return this._size === 0;
  }

  /**
   * Returns an array of all keys in the dictionary
   * Time Complexity: O(n)
   */
  keys()           {
    return Object.keys(this.items);
  }

  /**
   * Returns an array of all values in the dictionary
   * Time Complexity: O(n)
   */
  values()      {
    return Object.values(this.items);
  }

  /**
   * Iterates over each key-value pair in the dictionary
   * Time Complexity: O(n)
   * 
   * @param callbackFn - Function to execute for each entry
   */
  forEach(callbackFn                                 )       {
    for (const key in this.items) {
      if (Object.prototype.hasOwnProperty.call(this.items, key)) {
        callbackFn(this.items[key], key);
      }
    }
  }

  /**
   * Returns a string representation of the dictionary
   */
  toString()         {
    if (this.isEmpty()) {
      return '{}';
    }
    
    const pairs           = [];
    this.forEach((value, key) => {
      const valueStr = typeof value === 'string' ? `"${value}"` : String(value);
      pairs.push(`${key}: ${valueStr}`);
    });
    
    return `{ ${pairs.join(', ')} }`;
  }
}

// Example usage and demonstration
console.log('=== Dictionary (TypeScript) Demo ===\n');

// Create a dictionary with string values
const stringDict = new Dictionary        ();

console.log('1. Creating a Dictionary<string>');
stringDict.set('name', 'Alice');
stringDict.set('city', 'New York');
stringDict.set('country', 'USA');
console.log(`   Dictionary: ${stringDict.toString()}`);
console.log(`   Size: ${stringDict.size}`);
console.log();

// Create a dictionary with number values
const numberDict = new Dictionary        ();

console.log('2. Creating a Dictionary<number>');
numberDict.set('one', 1);
numberDict.set('two', 2);
numberDict.set('ten', 10);
console.log(`   Dictionary: ${numberDict.toString()}`);
console.log(`   Get "two": ${numberDict.get('two')}`);
console.log();

// Create a dictionary with object values
                    
                
              
 

const userDict = new Dictionary          ();

console.log('3. Creating a Dictionary<UserInfo>');
userDict.set('alice', { email: 'alice@example.com', age: 30 });
userDict.set('bob', { email: 'bob@example.com', age: 25 });

const alice = userDict.get('alice');
if (alice) {
  console.log(`   Alice's email: ${alice.email}`);
  console.log(`   Alice's age: ${alice.age}`);
}
console.log();

console.log('4. TypeScript typing note');
console.log('   In TypeScript, Dictionary<string> and Dictionary<number>');
console.log('   let the compiler check each dictionary\'s value type.');
console.log();

console.log('5. Iterating with forEach');
userDict.forEach((value, key) => {
  console.log(`   ${key}: ${value.email} (age: ${value.age})`);
});

// The Dictionary class is available for the examples in this cell.
})();

=== Dictionary (TypeScript) Demo ===



1. Creating a Dictionary<string>


   Dictionary: { name: "Alice", city: "New York", country: "USA" }


   Size: 3


2. Creating a Dictionary<number>


...

   In TypeScript, Dictionary<string> and Dictionary<number>


   let the compiler check each dictionary's value type.


5. Iterating with forEach


   alice: alice@example.com (age: 30)


   bob: bob@example.com (age: 25)


## Hash tables with linear probing

A dictionary maps keys to values; a hash table is one way to implement that mapping. A hash function turns a string key into an array index. Different keys can produce the same index, which is a collision. With **linear probing**, the table checks the next slot, then the next, until it finds the key or an empty slot. This contiguous run of occupied slots is called a cluster.

The example uses a simple “lose-lose” hash function: add the character codes, then take the remainder modulo 37. It is easy to understand, but many different strings can share a hash. The included `djb2` function is another hash choice; the public `hash` method currently selects lose-lose.

When removing a key, simply leaving a hole could make a later colliding key unreachable: lookup stops when it finds an empty slot. The implementation therefore rebuilds the following probe cluster after deletion. Expected put/get/remove time is O(1) when the table is lightly loaded; a long cluster can make them O(n). Hash quality and load factor affect performance.

### TypeScript implementation

The supplied TypeScript code uses an unbounded sparse array and mixes the array's changing `length` into probe wrapping. That can make a collision chain behave inconsistently as entries are inserted or deleted. The implementation below uses a fixed-capacity table with consistent wraparound and stops probing after checking every slot. It rebuilds a cluster after deletion. For simplicity it does not resize, so it reports failure when the table is full.

In [3]:
(() => {
                           
              
           
 

class HashTableLinearProbing    {
                   capacity = 1013;
          table                                  = new Array(this.capacity);
          _size = 0;

          loseLoseHashCode(key        )         {
    let hash = 0;
    for (let i = 0; i < key.length; i++) hash += key.charCodeAt(i);
    return hash % 37;
  }

          djb2HashCode(key        )         {
    let hash = 5381;
    for (let i = 0; i < key.length; i++) hash = (hash * 33 + key.charCodeAt(i)) >>> 0;
    return hash % 1013;
  }

  hash(key        )         { return this.loseLoseHashCode(key); }

  put(key        , value   )          {
    if (key == null || value == null) return false;
    let index = this.hash(key);
    for (let checked = 0; checked < this.capacity; checked++) {
      const entry = this.table[index];
      if (entry === undefined) {
        this.table[index] = { key, value };
        this._size++;
        return true;
      }
      if (entry.key === key) {
        entry.value = value;
        return true;
      }
      index = (index + 1) % this.capacity;
    }
    return false;
  }

  get(key        )                {
    let index = this.hash(key);
    for (let checked = 0; checked < this.capacity; checked++) {
      const entry = this.table[index];
      if (entry === undefined) return undefined;
      if (entry.key === key) return entry.value;
      index = (index + 1) % this.capacity;
    }
    return undefined;
  }

  remove(key        )          {
    let index = this.hash(key);
    for (let checked = 0; checked < this.capacity; checked++) {
      const entry = this.table[index];
      if (entry === undefined) return false;
      if (entry.key === key) {
        this.table[index] = undefined;
        this._size--;
        index = (index + 1) % this.capacity;
        while (this.table[index] !== undefined) {
          const displaced = this.table[index] ;
          this.table[index] = undefined;
          this._size--;
          this.put(displaced.key, displaced.value);
          index = (index + 1) % this.capacity;
        }
        return true;
      }
      index = (index + 1) % this.capacity;
    }
    return false;
  }

  get size()         { return this._size; }

  toString()         {
    const lines           = [];
    for (let i = 0; i < this.capacity; i++) {
      const entry = this.table[i];
      if (entry !== undefined) lines.push(`{${i}} => [${entry.key}: ${JSON.stringify(entry.value)}]`);
    }
    return lines.length ? lines.join('\n') : 'Hash table is empty';
  }

  getStats()                                                                          {
    let clusterCount = 0;
    let maxClusterSize = 0;
    let currentClusterSize = 0;
    for (let i = 0; i < this.capacity; i++) {
      if (this.table[i] !== undefined) {
        currentClusterSize++;
        maxClusterSize = Math.max(maxClusterSize, currentClusterSize);
      } else {
        if (currentClusterSize > 0) clusterCount++;
        currentClusterSize = 0;
      }
    }
    if (currentClusterSize > 0) clusterCount++;
    return { totalElements: this._size, clusterCount, maxClusterSize };
  }
}

console.log('=== Hash Table with Linear Probing (TypeScript) Demo ===\n');
const emailTable = new HashTableLinearProbing        ();
const contacts                     = [
  ['Ygritte', 'ygritte@email.com'], ['Jonathan', 'jonathan@email.com'],
  ['Jamie', 'jamie@email.com'], ['Jack', 'jack@email.com'],
  ['Nathan', 'nathan@email.com'], ['Sue', 'sue@email.com']
];
console.log('Hash values:');
for (const [name] of contacts) console.log(`  hash("${name}") = ${emailTable.hash(name)}`);
for (const [name, email] of contacts) emailTable.put(name, email);
console.log('\nTable contents:\n' + emailTable.toString());
console.log('\nJonathan lookup:', emailTable.get('Jonathan'));
console.log('Stats:', emailTable.getStats());
console.log('Remove Jonathan:', emailTable.remove('Jonathan'));
console.log('Jamie still found after cluster repair:', emailTable.get('Jamie'));
console.log('Sue still found after cluster repair:', emailTable.get('Sue'));
console.log('Updated size:', emailTable.size);
})();

=== Hash Table with Linear Probing (TypeScript) Demo ===



Hash values:


  hash("Ygritte") = 4


  hash("Jonathan") = 5


  hash("Jamie") = 5


...

Stats: { totalElements: 6, clusterCount: 2, maxClusterSize: 5 }


Remove Jonathan: true


Jamie still found after cluster repair: jamie@email.com


Sue still found after cluster repair: sue@email.com


Updated size: 5


### Original sparse-array implementation

The source version supplied for the book is represented by the runnable linear-probing implementation in the code cell above. The runnable version uses fixed capacity and consistent wraparound so probe sequences remain valid.

## Linked list dependency for separate chaining

The separate chaining hash table imports a `LinkedList` and uses its `append`, `removeAt`, `isEmpty`, `size`, and `toArray` methods to manage each bucket's collision chain. This is the TypeScript linked-list implementation supplied for that import. In a standalone project, save it as the module targeted by the hash table's import path, or include the class in the same module and remove the import line.

The generic `LinkedList<T>` stores nodes containing `T` values and provides operations such as insertion, removal, searching, mapping, filtering, and reduction. Its demo is guarded by `require.main === module`, which is intended for a Node.js file rather than a notebook cell.

### LinkedList implementation

The linked-list dependency and a small linked-list demonstration are included in the runnable code cell below, together with the separate-chaining table that uses it.

In [4]:
(() => {
/**
 * Linked List Implementation in TypeScript
 * 
 * A singly linked list where each node contains data and a reference to the next node.
 * This TypeScript version includes full type safety, generics, and interface definitions.
 */

/**
 * Interface for a basic linked list node
 * @template T The type of data stored in the node
 */
                              
          
                                  
 

/**
 * Interface for a linked list implementation
 * @template T The type of data stored in the list
 */
                          
                        
                         
                                             
                                
                                                                       
                                                                      
                     
                     
                
                     
                 
 

/**
 * LinkedListNode class representing a single node in the linked list
 * @template T The type of data stored in the node
 */
class LinkedListNode                                  {
         data   ;
         next                          ;

  /**
   * Creates a new linked list node
   * @param data - The data to store in this node
   * @param next - Reference to the next node (optional)
   */
  constructor(data   , next                           = null) {
    this.data = data;
    this.next = next;
  }
}

/**
 * Linked List Data Structure Implementation with TypeScript generics
 * 
 * A linear collection of data elements whose order is not given by their
 * physical placement in memory. Instead, each element points to the next.
 * 
 * @template T The type of data stored in the list
 * 
 * Real-world analogies:
 * - A chain where each link connects to the next link
 * - A treasure hunt where each clue leads to the next location
 * - A line of people where each person knows who's next
 * - A train where each car is connected to the next car
 * 
 * Advantages:
 * - Dynamic size - can grow and shrink during runtime
 * - Efficient insertion/deletion at the beginning (O(1))
 * - Memory efficient - only allocates memory when needed
 * - No memory waste - no unused allocated space
 * 
 * Disadvantages:
 * - No random access - must traverse from head to reach an element
 * - Extra memory overhead for storing pointers
 * - Not cache-friendly due to non-contiguous memory allocation
 * - Sequential access only
 */
class LinkedList                              {
          head                           = null;
          count         = 0;

  /**
   * Adds a new element to the end of the linked list
   * Time Complexity: O(n) - must traverse to the end
   * Space Complexity: O(1) - only creates one new node
   * 
   * @param data - The data to add
   */
         append(data   )       {
    const newNode = new LinkedListNode   (data);
    
    if (!this.head) {
      this.head = newNode;
    } else {
      let current = this.head;
      while (current.next) {
        current = current.next;
      }
      current.next = newNode;
    }
    this.count++;
  }

  /**
   * Adds a new element to the beginning of the linked list
   * Time Complexity: O(1) - direct insertion at head
   * Space Complexity: O(1) - only creates one new node
   * 
   * @param data - The data to add
   */
         prepend(data   )       {
    const newNode = new LinkedListNode   (data, this.head);
    this.head = newNode;
    this.count++;
  }

  /**
   * Inserts a new element at the specified position
   * Time Complexity: O(n) - may need to traverse to position
   * Space Complexity: O(1) - only creates one new node
   * 
   * @param data - The data to insert
   * @param position - The position to insert at (0-based)
   * @returns True if insertion was successful, false otherwise
   */
         insert(data   , position        )          {
    if (this.isInvalidPosition(position)) {
      return false;
    }
    
    if (position === 0) {
      this.prepend(data);
      return true;
    }
    
    const newNode = new LinkedListNode   (data);
    const previous = this.getNodeAt(position - 1);
    
    if (previous) {
      newNode.next = previous.next;
      previous.next = newNode;
      this.count++;
      return true;
    }
    
    return false;
  }

  /**
   * Removes and returns the element at the specified position
   * Time Complexity: O(n) - may need to traverse to position
   * Space Complexity: O(1) - no extra space used
   * 
   * @param position - The position of the element to remove (0-based)
   * @returns The removed element's data
   * @throws RangeError if position is invalid or list is empty
   */
         removeAt(position        )    {
    if (this.count === 0) {
      throw new RangeError('Cannot remove from an empty list.');
    }
    
    if (this.isInvalidPosition(position)) {
      throw new RangeError('Invalid position');
    }
    
    if (position === 0 && this.head) {
      const nodeToRemove = this.head;
      this.head = this.head.next;
      this.count--;
      return nodeToRemove.data;
    }
    
    const previous = this.getNodeAt(position - 1);
    if (previous && previous.next) {
      const nodeToRemove = previous.next;
      previous.next = nodeToRemove.next;
      this.count--;
      return nodeToRemove.data;
    }
    
    throw new Error('Unexpected error during removal');
  }

  /**
   * Removes the first occurrence of the specified data
   * Time Complexity: O(n) - may need to search entire list
   * Space Complexity: O(1) - no extra space used
   * 
   * @param data - The data to remove
   * @param compareFunction - Optional custom comparison function
   * @returns The removed element's data, or null if not found
   */
         remove(data   , compareFunction                          = (a, b) => a === b)           {
    const index = this.indexOf(data, compareFunction);
    if (index === -1) {
      return null;
    }
    return this.removeAt(index);
  }

  /**
   * Finds the position of the first occurrence of the specified data
   * Time Complexity: O(n) - may need to search entire list
   * Space Complexity: O(1) - no extra space used
   * 
   * @param data - The data to search for
   * @param compareFunction - Optional custom comparison function
   * @returns The index of the element, or -1 if not found
   */
         indexOf(data   , compareFunction                          = (a, b) => a === b)         {
    let current = this.head;
    let index = 0;
    
    while (current) {
      if (compareFunction(current.data, data)) {
        return index;
      }
      index++;
      current = current.next;
    }
    
    return -1;
  }

  /**
   * Checks if the linked list is empty
   * Time Complexity: O(1) - direct property access
   * Space Complexity: O(1) - no extra space used
   * 
   * @returns True if the list is empty, false otherwise
   */
         isEmpty()          {
    return this.count === 0;
  }

  /**
   * Returns the number of elements in the linked list
   * Time Complexity: O(1) - direct property access
   * Space Complexity: O(1) - no extra space used
   * 
   * @returns The number of elements
   */
         get size()         {
    return this.count;
  }

  /**
   * Removes all elements from the linked list
   * Time Complexity: O(1) - just resets head and count
   * Space Complexity: O(1) - no extra space used
   */
         clear()       {
    this.head = null;
    this.count = 0;
  }

  /**
   * Returns the element at the specified position without removing it
   * Time Complexity: O(n) - may need to traverse to position
   * Space Complexity: O(1) - no extra space used
   * 
   * @param position - The position to get the element from (0-based)
   * @returns The element at the specified position, or undefined if invalid position
   */
         get(position        )                {
    const node = this.getNodeAt(position);
    return node ? node.data : undefined;
  }

  /**
   * Converts the linked list to a string representation
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(n) - creates string with all elements
   * 
   * @returns String representation of the list
   */
         toString()         {
    if (!this.head) {
      return '';
    }
    
    let current = this.head;
    let objString = '';
    
    while (current) {
      objString += this.elementToString(current.data);
      if (current.next) {
        objString += ' -> ';
      }
      current = current.next;
    }
    
    return objString;
  }

  /**
   * Converts the linked list to an array
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(n) - creates new array with all elements
   * 
   * @returns Array containing all elements
   */
         toArray()      {
    const result      = [];
    let current = this.head;
    
    while (current) {
      result.push(current.data);
      current = current.next;
    }
    
    return result;
  }

  /**
   * Finds an element using a predicate function
   * Time Complexity: O(n) - may need to search entire list
   * Space Complexity: O(1) - no extra space used
   * 
   * @param predicate - Function that returns true for the desired element
   * @returns The first element that satisfies the predicate, or undefined if not found
   */
         find(predicate                      )                {
    let current = this.head;
    
    while (current) {
      if (predicate(current.data)) {
        return current.data;
      }
      current = current.next;
    }
    
    return undefined;
  }

  /**
   * Filters elements using a predicate function
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(k) - where k is number of matching elements
   * 
   * @param predicate - Function that returns true for elements to include
   * @returns New LinkedList containing only elements that satisfy the predicate
   */
         filter(predicate                      )                {
    const result = new LinkedList   ();
    let current = this.head;
    
    while (current) {
      if (predicate(current.data)) {
        result.append(current.data);
      }
      current = current.next;
    }
    
    return result;
  }

  /**
   * Transforms elements using a mapping function
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(n) - creates new list with transformed elements
   * 
   * @template U The type of the transformed elements
   * @param mapper - Function that transforms each element
   * @returns New LinkedList containing transformed elements
   */
         map   (mapper                )                {
    const result = new LinkedList   ();
    let current = this.head;
    
    while (current) {
      result.append(mapper(current.data));
      current = current.next;
    }
    
    return result;
  }

  /**
   * Executes a function for each element in the list
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(1) - no extra space used
   * 
   * @param callback - Function to execute for each element
   */
         forEach(callback                                  )       {
    let current = this.head;
    let index = 0;
    
    while (current) {
      callback(current.data, index);
      current = current.next;
      index++;
    }
  }

  /**
   * Reduces the list to a single value using a reducer function
   * Time Complexity: O(n) - must traverse entire list
   * Space Complexity: O(1) - no extra space used (excluding accumulator)
   * 
   * @template U The type of the accumulated value
   * @param reducer - Function that combines elements
   * @param initialValue - Initial value for the accumulator
   * @returns The accumulated result
   */
         reduce   (reducer                                                  , initialValue   )    {
    let current = this.head;
    let accumulator = initialValue;
    let index = 0;
    
    while (current) {
      accumulator = reducer(accumulator, current.data, index);
      current = current.next;
      index++;
    }
    
    return accumulator;
  }

  // Private helper methods

  /**
   * Gets the node at the specified position
   * @private
   * @param position - The position to get the node from
   * @returns The node at the position, or null if invalid
   */
          getNodeAt(position        )                           {
    if (this.isInvalidPosition(position)) {
      return null;
    }
    
    let current = this.head;
    for (let index = 0; index < position && current; index++) {
      current = current.next;
    }
    
    return current;
  }

  /**
   * Validates if a position is within valid bounds
   * @private
   * @param position - The position to validate
   * @returns True if position is invalid
   */
          isInvalidPosition(position        )          {
    return position < 0 || position > this.count;
  }

  /**
   * Converts an element to its string representation
   * @private
   * @param element - The element to convert
   * @returns String representation of the element
   */
          elementToString(element   )         {
    if (element === null) {
      return 'null';
    } else if (element === undefined) {
      return 'undefined';
    } else if (typeof element === 'string') {
      return `"${element}"`;
    } else {
      return String(element);
    }
  }
}

/**
 * HashTable with Separate Chaining for Collision Handling (TypeScript)
 * 
 * Separate chaining stores colliding elements in linked lists at each bucket.
 * This TypeScript implementation provides full type safety.
 * 
 * @template V The type of values stored in the hash table
 */


/**
 * Interface for key-value pairs stored in the hash table
 */
                           
              
           
 

class HashTableSeparateChaining    {
  // Private array where each position holds a linked list
          table                                              = [];

  /**
   * The "lose-lose" hash function
   * @private
   */
          loseLoseHashCode(key        )         {
    let hash = 0;
    for (let i = 0; i < key.length; i++) {
      hash += key.charCodeAt(i);
    }
    return hash % 37;
  }

  /**
   * The "djb2" hash function - better distribution
   * @private
   */
          djb2HashCode(key        )         {
    let hash = 5381;
    for (let i = 0; i < key.length; i++) {
      hash = (hash * 33) + key.charCodeAt(i);
    }
    return hash % 1013;
  }

  /**
   * Public hash function wrapper
   */
  hash(key        )         {
    return this.loseLoseHashCode(key);
  }

  /**
   * Finds the index of a key within a linked list at a given bucket
   * @private
   */
          findKeyIndex(bucketIndex        , key        )         {
    const linkedList = this.table[bucketIndex];
    if (linkedList == null) {
      return -1;
    }
    
    const items = linkedList.toArray();
    for (let i = 0; i < items.length; i++) {
      if (items[i].key === key) {
        return i;
      }
    }
    
    return -1;
  }

  /**
   * Adds or updates a key-value pair in the hash table
   * Time Complexity: O(1) average, O(n) worst case
   */
  put(key        , value   )          {
    if (key != null && value != null) {
      const index = this.hash(key);
      
      // Create a new linked list at this index if needed
      if (this.table[index] == null) {
        this.table[index] = new LinkedList                 ();
      }
      
      // Check if key already exists and update it
      const existingIndex = this.findKeyIndex(index, key);
      if (existingIndex >= 0) {
        this.table[index] .removeAt(existingIndex);
      }
      
      // Add the key-value pair to the linked list
      this.table[index] .append({ key, value });
      return true;
    }
    return false;
  }

  /**
   * Retrieves the value associated with a key
   * Time Complexity: O(1) average, O(n) worst case
   */
  get(key        )                {
    const index = this.hash(key);
    const linkedList = this.table[index];
    
    if (linkedList != null) {
      const items = linkedList.toArray();
      for (const item of items) {
        if (item.key === key) {
          return item.value;
        }
      }
    }
    
    return undefined;
  }

  /**
   * Removes the key-value pair associated with a key
   * Time Complexity: O(1) average, O(n) worst case
   */
  remove(key        )          {
    const index = this.hash(key);
    const linkedList = this.table[index];
    
    if (linkedList != null) {
      const keyIndex = this.findKeyIndex(index, key);
      
      if (keyIndex >= 0) {
        linkedList.removeAt(keyIndex);
        
        // Clean up empty linked lists
        if (linkedList.isEmpty()) {
          this.table[index] = undefined;
        }
        
        return true;
      }
    }
    
    return false;
  }

  /**
   * Returns a string representation of the hash table
   */
  toString()         {
    const keys = Object.keys(this.table);
    
    if (keys.length === 0) {
      return 'Hash table is empty';
    }
    
    let result = '';
    for (const bucketIndex of keys) {
      const linkedList = this.table[parseInt(bucketIndex)];
      if (linkedList != null) {
        const items = linkedList.toArray();
        const itemsStr = items.map(item => `[${item.key}: ${item.value}]`).join(' -> ');
        result += `{Bucket ${bucketIndex}} => ${itemsStr}\n`;
      }
    }
    
    return result.trim();
  }

  /**
   * Returns statistics about the hash table
   */
  getStats()                                                                                                 {
    let totalElements = 0;
    let maxChainLength = 0;
    let bucketsUsed = 0;
    
    for (const index in this.table) {
      const linkedList = this.table[index];
      if (linkedList != null && !linkedList.isEmpty()) {
        bucketsUsed++;
        const chainLength = linkedList.size;
        totalElements += chainLength;
        maxChainLength = Math.max(maxChainLength, chainLength);
      }
    }
    
    return {
      totalElements,
      bucketsUsed,
      maxChainLength,
      avgChainLength: bucketsUsed > 0 ? (totalElements / bucketsUsed).toFixed(2) : '0'
    };
  }
}

// Example usage and demonstration
console.log('=== HashTable with Separate Chaining (TypeScript) Demo ===\n');

// Create hash table with string values
const emailTable = new HashTableSeparateChaining        ();

console.log('1. Adding contacts with colliding hash values');

const contacts                     = [
  ['Ygritte', 'ygritte@email.com'],
  ['Jonathan', 'jonathan@email.com'],
  ['Jamie', 'jamie@email.com'],
  ['Jack', 'jack@email.com'],
  ['Nathan', 'nathan@email.com'],
  ['Sue', 'sue@email.com'],
];

console.log('   Hash values:');
for (const [name] of contacts) {
  console.log(`   hash("${name}") = ${emailTable.hash(name)}`);
}
console.log();

console.log('2. Adding all contacts...');
for (const [name, email] of contacts) {
  emailTable.put(name, email);
}
console.log(emailTable.toString().split('\n').map(line => '   ' + line).join('\n'));
console.log();

console.log('3. Retrieving values with type safety');
const jonathanEmail                     = emailTable.get('Jonathan');
console.log(`   Jonathan's email: ${jonathanEmail}`);
console.log(`   Sue's email: ${emailTable.get('Sue')}`);
console.log();

console.log('4. Statistics');
const stats = emailTable.getStats();
console.log(`   Total elements: ${stats.totalElements}`);
console.log(`   Buckets used: ${stats.bucketsUsed}`);
console.log(`   Max chain length: ${stats.maxChainLength}`);
console.log(`   Avg chain length: ${stats.avgChainLength}`);

const names = new LinkedList();
names.append('Ada');
names.append('Grace');
console.log('Linked list:', names.toString(), '| size:', names.size);
})();


=== HashTable with Separate Chaining (TypeScript) Demo ===



1. Adding contacts with colliding hash values


   Hash values:


   hash("Ygritte") = 4


   hash("Jonathan") = 5


...

   Total elements: 6


   Buckets used: 4


   Max chain length: 3


   Avg chain length: 1.50


Linked list: "Ada" -> "Grace" | size: 2


## Hash tables with separate chaining

Separate chaining handles collisions by keeping a linked list at each hash bucket. When keys map to the same index, their key-value pairs are appended to that bucket's list. Lookup and removal first choose a bucket with the hash function, then search that list. Average performance depends on keys spreading across buckets; a long chain can make an operation O(n).

This TypeScript example uses the `LinkedList` class from the linked-list chapter, so its import path must resolve in the project for the code to compile. The generic parameter `V` keeps stored values and returned values consistently typed. The statistics report how many buckets are occupied and the longest and average chain lengths.

### Separate chaining implementation

The runnable cell below includes the linked-list dependency, the generic separate-chaining hash table, and examples for insertion, lookup, removal, and chain statistics.

## Basic hash table without collision handling

This example hashes a string key to an array index and stores the value at that index. The generic `HashTable<V>` keeps values consistently typed, and its `put`, `get`, and `remove` methods are short because they do not search for a key after hashing.

This is a teaching step, not a safe general-purpose map: if two different keys produce the same index, the later `put` overwrites the earlier value, and `get` cannot distinguish them. In this example, `Gandalf` and `Tyrion` both hash to index 19 with the lose-lose hash function, so storing both loses one entry. The linear probing and separate chaining examples later in the notebook show ways to handle collisions.

In [5]:
(() => {
/**
 * HashTable Data Structure Implementation in TypeScript
 * 
 * A hash table (hash map) stores key-value pairs using a hash function
 * to determine storage locations. This TypeScript implementation
 * provides type safety through generics.
 * 
 * Note: This basic implementation does NOT handle collisions.
 * See hash-table-separate-chaining.ts and hash-table-linear-probing.ts
 * for collision-resistant implementations.
 * 
 * @template V The type of values stored in the hash table
 */

class HashTable    {
  // Private array to store values
          table      = [];

  /**
   * Converts any data type to a string representation
   * @private
   */
          elementToString(data         )         {
    if (data === null) {
      return 'null';
    } else if (data === undefined) {
      return 'undefined';
    } else if (typeof data === 'object') {
      return JSON.stringify(data);
    } else {
      return String(data);
    }
  }

  /**
   * The "lose-lose" hash function
   * Simple but collision-prone - good for demonstration
   * @private
   */
          loseLoseHashCode(key        )         {
    let hash = 0;
    for (let i = 0; i < key.length; i++) {
      hash += key.charCodeAt(i);
    }
    return hash % 37;
  }

  /**
   * The "djb2" hash function - a better alternative
   * @private
   */
          djb2HashCode(key        )         {
    let hash = 5381;
    for (let i = 0; i < key.length; i++) {
      hash = (hash * 33) + key.charCodeAt(i);
    }
    return hash % 1013;
  }

  /**
   * Public hash function wrapper
   * In TypeScript, the key parameter is typed as string,
   * eliminating the need for type checking inside the function
   */
  hash(key        )         {
    return this.loseLoseHashCode(key);
    // Use this for better distribution:
    // return this.djb2HashCode(key);
  }

  /**
   * Adds or updates a key-value pair in the hash table
   * Time Complexity: O(1) average case
   * 
   * @param key - The string key to set
   * @param value - The value to store
   * @returns True if successful, false if key or value is invalid
   */
  put(key        , value   )          {
    if (key == null || value == null) {
      return false;
    }
    
    const index = this.hash(key);
    this.table[index] = value;
    return true;
  }

  /**
   * Retrieves the value associated with a key
   * Time Complexity: O(1) average case
   * 
   * @param key - The string key to look up
   * @returns The value, or undefined if not found
   */
  get(key        )                {
    if (key == null) {
      return undefined;
    }
    
    const index = this.hash(key);
    return this.table[index];
  }

  /**
   * Removes the value associated with a key
   * Time Complexity: O(1) average case
   * 
   * @param key - The string key to remove
   * @returns True if found and removed, false otherwise
   */
  remove(key        )          {
    if (key == null) {
      return false;
    }
    
    const index = this.hash(key);
    if (this.table[index] != null) {
      delete this.table[index];
      return true;
    }
    return false;
  }

  /**
   * Returns a string representation of the hash table
   */
  toString()         {
    const keys = Object.keys(this.table);
    
    if (keys.length === 0) {
      return 'Hash table is empty';
    }
    
    let objString = '';
    for (const key of keys) {
      const value = this.table[parseInt(key)];
      const valueStr = typeof value === 'string' ? `"${value}"` : String(value);
      objString += `{${key} => ${valueStr}}\n`;
    }
    
    return objString.trim();
  }

  /**
   * Returns a copy of the underlying table for inspection
   */
  getTable()      {
    return [...this.table];
  }
}

// Example usage and demonstration
console.log('=== HashTable (TypeScript) Demo ===\n');

// Create a hash table with string values (email addresses)
const emailTable = new HashTable        ();

console.log('1. Creating a HashTable<string> for email addresses');
console.log('   Computing hash codes:');
console.log(`   hash("Gandalf") = ${emailTable.hash('Gandalf')}`);
console.log(`   hash("John") = ${emailTable.hash('John')}`);
console.log(`   hash("Tyrion") = ${emailTable.hash('Tyrion')}`);
console.log();

console.log('2. Adding contacts');
emailTable.put('Gandalf', 'gandalf@email.com');
emailTable.put('John', 'johnsnow@email.com');
emailTable.put('Tyrion', 'tyrion@email.com');
console.log(emailTable.toString().split('\n').map(line => '   ' + line).join('\n'));
console.log();

console.log('3. Retrieving values with type safety');
const gandalfEmail                     = emailTable.get('Gandalf');
console.log(`   Gandalf's email: ${gandalfEmail}`);
console.log(`   Unknown user: ${emailTable.get('Unknown')}`);
console.log();

// Create a hash table with number values
const scoreTable = new HashTable        ();

console.log('4. Creating a HashTable<number> for scores');
scoreTable.put('Alice', 95);
scoreTable.put('Bob', 87);
scoreTable.put('Charlie', 92);

const aliceScore                     = scoreTable.get('Alice');
if (aliceScore !== undefined) {
  console.log(`   Alice's score: ${aliceScore}`);
  console.log(`   Is passing (>= 60): ${aliceScore >= 60}`);
}
console.log();

// Create a hash table with object values
                       
               
                
                                   
 

const userTable = new HashTable             ();

console.log('5. Creating a HashTable<UserProfile> with typed objects');
userTable.put('admin', { name: 'Admin User', email: 'admin@example.com', role: 'admin' });
userTable.put('alice', { name: 'Alice Smith', email: 'alice@example.com', role: 'user' });

const adminUser = userTable.get('admin');
if (adminUser) {
  // TypeScript knows adminUser has type UserProfile
  console.log(`   Admin name: ${adminUser.name}`);
  console.log(`   Admin role: ${adminUser.role}`);
  console.log(`   Admin email: ${adminUser.email}`);
}
console.log();

console.log('6. TypeScript typing note:');
console.log('   The original TypeScript signature constrains keys to strings');
console.log('   and uses a generic V to type stored values at compile time.');

// HashTable is used by the demonstrations in this cell.
})();

=== HashTable (TypeScript) Demo ===



1. Creating a HashTable<string> for email addresses


   Computing hash codes:


   hash("Gandalf") = 19


   hash("John") = 29


...

   Admin role: admin


   Admin email: admin@example.com


6. TypeScript typing note:


   The original TypeScript signature constrains keys to strings


   and uses a generic V to type stored values at compile time.


## Book chapter companion: dictionaries and hashes

A set stores distinct values; a dictionary associates a unique key with a value. Hash tables use a hash function to choose where a key-value pair is stored. In everyday JavaScript and TypeScript, native `Map` is often the simplest dictionary. Custom hash tables help explain how lookup works and how collisions must be handled.

Dictionaries are also called maps, dictionaries, associative arrays, or symbol tables in different languages and contexts. They are widely used for caches, indexes, configuration, frequency counts, and fast lookup. The custom dictionary and hash table implementations above demonstrate the underlying mechanics.

### AI response cache and exact-key hashing

A cache can map an exact request to a previously generated response. For large requests, a cryptographic digest can serve as a compact key. Hashing does not compare meaning: two semantically similar prompts with different text produce different keys. Similarity search requires other techniques, such as embeddings and vector search.



**Runnable example follows in the next code cell.**


This example requires Node.js because it imports `node:crypto`. A production cache may also need expiration, size limits, and request configuration included in its key.

In [6]:
(() => {
const { createHash } = require('node:crypto');
const responseCache = new Map                ();

function hashPrompt(prompt        )         {
  return createHash('sha256').update(prompt).digest('hex');
}

function cacheResponse(prompt        , response        )       {
  responseCache.set(hashPrompt(prompt), response);
}

function getCachedResponse(prompt        )                     {
  return responseCache.get(hashPrompt(prompt));
}

const prompt = 'Explain how a binary tree works.';
cacheResponse(prompt, 'A binary tree has nodes with at most two children.');
const cached = getCachedResponse(prompt);
if (cached !== undefined) console.log('Using cached response:', cached);
})();

Using cached response: A binary tree has nodes with at most two children.


### Vocabulary maps and native collection types

A vocabulary can map tokens to numeric IDs and convert a token sequence to IDs. The lookup result is optional because a token may not be present.



**Runnable example follows in the next code cell.**


Language models commonly tokenize text into words, word parts, punctuation, or other units. This map illustrates the key-value relationship without modeling a complete tokenizer.

`Map<K, V>` supports keys of any type and preserves key identity. `WeakMap` and `WeakSet` accept object keys or members and do not keep those objects alive solely because they are in the collection. They are not enumerable, so they do not expose all keys or values for iteration.



**Runnable example follows in the next code cell.**


Use `Map` when you need arbitrary keys and iteration. Use `WeakMap` or `WeakSet` for object-associated metadata when weak references and non-enumerability are appropriate.

In [7]:
(() => {
const vocabulary = new Map                ([
  ['hello', 101], ['world', 102], ['computer', 103], ['learning', 104]
]);

const sentence = ['hello', 'world'];
const tokenIds = sentence.map((word) => vocabulary.get(word));
console.log(tokenIds); // [101, 102]
})();

[ 101, 102 ]


In [8]:
(() => {
const objectMetadata = new WeakMap                              ();
const processedObjects = new WeakSet        ();
const node = { id: 1 };
objectMetadata.set(node, { visited: true });
processedObjects.add(node);
console.log(objectMetadata.get(node)?.visited); // true
console.log(processedObjects.has(node)); // true
})();

true


true


### Hash sets, efficiency, and choosing a structure

A hash set stores unique values and uses hashing to support average O(1) membership checks, insertion, and removal. JavaScript's native `Set` provides this behavior without requiring a custom implementation.

| Structure | Typical average lookup/update | Worst case or trade-off |
| --- | --- | --- |
| Dictionary / native `Map` | O(1) | Depends on the runtime implementation and key behavior |
| Basic hash table without collision handling | O(1) | Collisions overwrite values and lose data |
| Separate chaining | O(1) | O(n) if many keys land in one bucket; linked-list overhead |
| Linear probing | O(1) at low load | O(n) with clustering or a high load factor |

The **load factor** is `number of stored entries / number of buckets`. As it rises, collisions and probe lengths tend to increase. Resizing and rehashing can help preserve performance. In general, use native `Map` or plain object properties for ordinary application storage; build a custom collision-handling table when learning or when you need to control its behavior.

### Practice and chapter review

The Integer to Roman problem uses a descending dictionary of numeral tokens and values. Repeatedly choose the largest token that fits the remaining number. The TypeScript implementations near the start of this notebook demonstrate the object-dictionary and native `Map` variants.

Other exercises that practice these ideas include:

- **Two Sum:** map each seen number to its index to find complements quickly.
- **Valid Anagram:** count character frequencies with a map.
- **Design HashSet / Design HashMap:** implement storage and collision handling.
- **Roman to Integer:** scan symbols and account for subtractive pairs.

This chapter connects key-value storage, hashing, collision resolution, and TypeScript generics. The key choice is often between native `Map` for application code and a custom hash table when the implementation itself is the subject.